# Early FTUE analysis

**Purpose:** 

**Data range:** 

---



In [142]:
# show → code input visible by default
# hide-output → output hidden by default
# show hide-output → both (can combine on one line)

# Standard data analysis stack + project utilities
import pandas as pd
from common_lib.sql import BigQueryConnector
from common_lib.export import export_notebook_html
import plotly.express as px
import numpy as np

bqc = BigQueryConnector()

## Aux functions

In [143]:
def compute_weighted_progression(data, measure_col, dimension_cols=['install_dt', 'days_since_install'], min_bucket_size=50):
    """
    Compute weighted average progression metric for a given measure across dimensions.
    
    Parameters:
    -----------
    data : pd.DataFrame
        Source data containing user_id, the measure column, and dimension columns
    measure_col : str
        Column name to compute weighted average for (e.g., 'max_level', 'max_gameday')
    dimension_cols : list
        Dimensions to group by (default: ['install_dt', 'days_since_install'])
    min_bucket_size : int
        Minimum users per bucket to include (default: 50)
    
    Returns:
    --------
    pd.DataFrame
        Aggregated data with weighted average and cohort user counts
    """
    
    # Step 1: Count unique users per dimension + measure bucket
    agg = data.groupby(dimension_cols + [measure_col]).agg(
        unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 2: Total unique users per dimension combination
    dimension_total_users = data.groupby(dimension_cols).agg(
        total_unique_users=('user_id', 'nunique')
    ).reset_index()
    
    # Step 3: Merge and compute percentage share
    agg = agg.merge(dimension_total_users, on=dimension_cols)
    agg['percentage_of_users'] = agg['unique_users'] / agg['total_unique_users']
    
    # Step 4: Compute weighted average
    weighted_avg = agg.groupby(dimension_cols, group_keys=False).apply(
        lambda x: (x[measure_col] * x['unique_users']).sum() / x['unique_users'].sum(),
        include_groups=False
    ).reset_index()
    
    weighted_avg.columns = dimension_cols + [f'weighted_avg_{measure_col}']
    agg = agg.merge(weighted_avg, on=dimension_cols)
    
    # Step 5: Drop small buckets
    agg = agg[agg['unique_users'] >= min_bucket_size]
    
    # Step 6: Collapse to one row per dimension combination
    agg = agg.groupby(dimension_cols).agg(
        cohort_users=('unique_users', 'sum'),
        **{f'weighted_avg_{measure_col}': ('weighted_avg_' + measure_col, 'first')}
    ).reset_index()
    
    return agg

def weighted_quantiles(group, quantiles=[0.1, 0.5, 0.9], measure_col='max_level'):
    """Return P10 / P50 / P90 of measure_col, using user counts as weights.

    Sorts by the measure, accumulates weights, then uses searchsorted to find
    the value at each quantile threshold — equivalent to a weighted percentile.
    """
    levels = group[measure_col].values
    weights = group['users'].values
    sorted_idx = np.argsort(levels)
    levels, weights = levels[sorted_idx], weights[sorted_idx]
    cum_weights = np.cumsum(weights)
    total = cum_weights[-1]
    result = {}
    for q in quantiles:
        idx = np.searchsorted(cum_weights, q * total)
        result[f'p{int(q * 100)}'] = levels[min(idx, len(levels) - 1)]
    return pd.Series(result)



def add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=None):
    """
    Add vertical line annotations for events to a plotly figure with interactive toggles.
    Offsets annotation text positions to prevent overlap.
    """
    
    for ftue_type, events in events_config.items():
        for i, event in enumerate(events):
            # Alternate text position: top for A.old, top left for B.new
            text_position = 'top left' if ftue_type == 'B.new' else 'top'
            
            fig.add_vline(
                x=event['level'],
                line_dash='dash',
                line_color=event['color'],
                opacity=0.7,
                name=ftue_type,
                legendgroup=ftue_type,
                #showlegend=(i == 0),
                showlegend = True,
                annotation_text=event['name'],
                annotation_position=text_position,
            )
    
    return fig

## Get data

### Player level and game day

In [190]:
# calculate the start date based on the number of days from 2026-06-01 to today

import datetime as dt

new_ftue_date = dt.datetime(2026, 6, 1)
days_from_start = (dt.datetime.today() - new_ftue_date).days+3
start_date1 = new_ftue_date - dt.timedelta(days=days_from_start)
end_date1 = new_ftue_date-dt.timedelta(days=1)
start_date2 = new_ftue_date
end_date2 = dt.datetime.today()

# print all dates
print(f"Start Date 1: {start_date1.strftime('%Y-%m-%d')}")
print(f"End Date 1: {end_date1.strftime('%Y-%m-%d')}")
print(f"Start Date 2: {start_date2.strftime('%Y-%m-%d')}")
print(f"End Date 2: {end_date2.strftime('%Y-%m-%d')}")

Start Date 1: 2026-05-08
End Date 1: 2026-05-31
Start Date 2: 2026-06-01
End Date 2: 2026-06-22


In [191]:
refresh_data = True

In [192]:
# Point to the SQL file and set the start date for the cohort window

query_location = './sql/playerlevel.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 17.09 GB when run.
Estimated query cost: $0.11


In [193]:
data = pd.DataFrame()

# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    data = bqc.get(query='./sql/playerlevel.sql', is_path=True, query_parameters=parameters)
    data.to_pickle('./data/playprogression.pkl')
else:
    # Load from local cache to avoid repeated query costs
    data = pd.read_pickle('./data/playprogression.pkl')

In [194]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,install_build_version
0,F0BF22A8C4EE886C,2026-05-31,2026-05-28,2026-05-24,2026-05-01,3,7,4,CPE,0.75.0
1,F4E7B68973C9B8C9,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,4,2,CPE,0.75.0
2,9446E454237203C7,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,5,3,CPE,0.75.0
3,FB37506040A8CE5F,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,2,1,CPE,0.75.0
4,A8514349B271C50E,2026-05-31,2026-05-29,2026-05-24,2026-05-01,2,7,4,Non-Attributed,0.75.0
...,...,...,...,...,...,...,...,...,...,...
164819,F574F122CA81CCA6,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,6,4,CPE,0.77.0
164820,EA26E8DAA8001B5D,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,3,1,UA,0.77.0
164821,6608069738F8620F,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,4,2,Non-Attributed,0.77.0
164822,AF6C23ED8DE9D62,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,5,3,CPE,0.77.0


### Retention

In [195]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retention.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [227]:
retention_data = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data = bqc.get(query='./sql/retention.sql', is_path=True, query_parameters=parameters)
    retention_data.to_pickle('./data/retention.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data = pd.read_pickle('./data/retention.pkl')

In [197]:
retention_data.sort_values(['install_dt', 'dx','platform'], inplace=True)
retention_data[retention_data['platform'] == 'AND']

,install_dt,dx,platform,cohort_size,retained_size,retention_rate
0,2026-05-08,0,AND,369,369,1.000000
2,2026-05-08,1,AND,369,154,0.417344
5,2026-05-08,3,AND,369,116,0.314363
7,2026-05-08,7,AND,369,96,0.260163
8,2026-05-08,14,AND,369,63,0.170732
...,...,...,...,...,...,...
347,2026-06-19,0,AND,461,461,1.000000
348,2026-06-19,1,AND,461,180,0.390456
351,2026-06-20,0,AND,477,477,1.000000
353,2026-06-20,1,AND,477,204,0.427673


In [198]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retentiontotal.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [233]:
retention_data_total = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache

if refresh_data:
    retention_data_total = bqc.get(query='./sql/retentiontotal.sql', is_path=True, query_parameters=parameters)
    retention_data_total.to_pickle('./data/retentiontotal.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total = pd.read_pickle('./data/retentiontotal.pkl')

In [237]:
retention_data_total.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total[retention_data_total['dx'] == 14]

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
17,14,AND,A.Pre-FTUE revamp,24,7627,608,0.079717
16,14,AND,B.Post-FTUE revamp,21,5795,493,0.085073
18,14,IOS,A.Pre-FTUE revamp,22,16035,1254,0.078204
19,14,IOS,B.Post-FTUE revamp,21,9554,932,0.097551


In [201]:
# Point to the SQL file and set the start date for the cohort window
query_location = './sql/retentiontotalNA.sql'
parameters = {
    'start_date1': start_date1.strftime('%Y-%m-%d'),
    'end_date1': end_date1.strftime('%Y-%m-%d'),
    'start_date2': start_date2.strftime('%Y-%m-%d'),
    'end_date2': end_date2.strftime('%Y-%m-%d'),
}

# Print cost estimate before running — avoids accidental expensive queries (~$1.63 / 242 GB)
cost_info = bqc.print_cost_estimate(query=query_location, is_path=True, query_parameters=parameters)

This query will process 1.29 GB when run.
Estimated query cost: $0.01


In [202]:
retention_data_total_na = pd.DataFrame()
# Toggle to True to re-run the BQ query and overwrite the local cache
if refresh_data:
    retention_data_total_na = bqc.get(query='./sql/retentiontotalNA.sql', is_path=True, query_parameters=parameters)
    retention_data_total_na.to_pickle('./data/retentiontotalNA.pkl')
else:
    # Load from local cache to avoid repeated query costs
    retention_data_total_na = pd.read_pickle('./data/retentiontotalNA.pkl')

In [203]:
retention_data_total_na.sort_values([ 'dx', 'platform','FTUE_flag'], inplace=True)
retention_data_total_na

,dx,platform,FTUE_flag,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,7627,7627,1.000000
0,0,AND,B.Post-FTUE revamp,9024,9024,1.000000
3,0,IOS,A.Pre-FTUE revamp,16035,16035,1.000000
2,0,IOS,B.Post-FTUE revamp,17815,17815,1.000000
5,1,AND,A.Pre-FTUE revamp,7627,2994,0.392553
4,1,AND,B.Post-FTUE revamp,9012,3293,0.365402
6,1,IOS,A.Pre-FTUE revamp,16035,6624,0.413096
7,1,IOS,B.Post-FTUE revamp,17812,6900,0.387379
8,3,AND,A.Pre-FTUE revamp,7627,2066,0.270880
9,3,AND,B.Post-FTUE revamp,9012,1932,0.214381


## Process data

In [204]:
dt_mode = 'install_dt'

data['install_dt'] = data[dt_mode]

data['CPE_flag'] = ['Y' if x == 'CPE' else 'N' for x in data['acquisition_type']]

data['FTUE_flag'] = ['B.new' if x >='0.76.0' else 'A.old' for x in data['install_build_version']]

# Making comparison fair
max_dayx_B_new = (pd.to_datetime('today') - pd.to_datetime('2026-06-01')).days
data = data[~(data['days_since_install'] > max_dayx_B_new)]

data.loc[:,'dummy'] = 'dummy'

# Require each cohort to have had enough calendar time to be meaningful:
# weekly cohorts need 7 days, monthly cohorts need 30.
min_days_since_install = 0
if dt_mode == 'install_dt_week':
    min_days_since_install = 7
elif dt_mode == 'install_dt_month':
    min_days_since_install = 30 

# Drop rows where the player's most recent observed day falls within the cohort's maturity window.
# This prevents partially-observed cohorts from pulling down progression averages.
data = data[data['days_since_install'] <= (pd.to_datetime('today') - pd.to_datetime(data['install_dt'])).dt.days - min_days_since_install]

data

/var/folders/vj/nwmxdwwn0hl5rtj_xcywvvpc0000gn/T/ipykernel_3522/2906483755.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data.loc[:,'dummy'] = 'dummy'


,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,F0BF22A8C4EE886C,2026-05-31,2026-05-28,2026-05-24,2026-05-01,3,7,4,CPE,0.75.0,Y,A.old,dummy
1,F4E7B68973C9B8C9,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,4,2,CPE,0.75.0,Y,A.old,dummy
2,9446E454237203C7,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,5,3,CPE,0.75.0,Y,A.old,dummy
3,FB37506040A8CE5F,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,2,1,CPE,0.75.0,Y,A.old,dummy
4,A8514349B271C50E,2026-05-31,2026-05-29,2026-05-24,2026-05-01,2,7,4,Non-Attributed,0.75.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164819,F574F122CA81CCA6,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,6,4,CPE,0.77.0,Y,B.new,dummy
164820,EA26E8DAA8001B5D,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,3,1,UA,0.77.0,N,B.new,dummy
164821,6608069738F8620F,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,4,2,Non-Attributed,0.77.0,N,B.new,dummy
164822,AF6C23ED8DE9D62,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,5,3,CPE,0.77.0,Y,B.new,dummy


In [205]:
test = data.groupby(['FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

test

,FTUE_flag,users
0,A.old,17864
1,B.new,19550


## Player Level reached at day x

For each install cohort, tracks the **weighted average max level** reached as a function of days since install. Weighted average is used to account for varying player counts across level buckets. The percentage-change chart below highlights where the steepest level gains occur in the early-day window.

In [206]:
pl_ftue_max_level_agg = compute_weighted_progression(data, measure_col='max_level', dimension_cols=['dummy', 'days_since_install','FTUE_flag'], min_bucket_size=50)
pl_ftue_max_level_agg['combined_dimension'] = pl_ftue_max_level_agg['dummy'].astype(str) + ' | ' + pl_ftue_max_level_agg['FTUE_flag']

pl_ftue_max_level_agg['pctg_diff'] = pl_ftue_max_level_agg.groupby('days_since_install')['weighted_avg_max_level'].pct_change().fillna(0)

pl_ftue_total_users = pl_ftue_max_level_agg[pl_ftue_max_level_agg['days_since_install'] == 0][['FTUE_flag', 'cohort_users']].rename(columns={'cohort_users': 'total_cohort_users'})
pl_ftue_max_level_agg = pl_ftue_max_level_agg.merge(pl_ftue_total_users, on=['FTUE_flag'], how='left')
pl_ftue_max_level_agg['pctg_users'] = pl_ftue_max_level_agg['cohort_users'] / pl_ftue_max_level_agg['total_cohort_users']
pl_ftue_max_level_agg['pctg_diff_users'] = pl_ftue_max_level_agg.groupby('days_since_install')['pctg_users'].pct_change().fillna(0)

pl_ftue_max_level_agg


,dummy,days_since_install,FTUE_flag,cohort_users,weighted_avg_max_level,combined_dimension,pctg_diff,total_cohort_users,pctg_users,pctg_diff_users
0,dummy,0,A.old,17669,3.621509,dummy | A.old,0.000000,17669,1.000000,0.000000
1,dummy,0,B.new,19404,3.654821,dummy | B.new,0.009198,19404,1.000000,0.000000
2,dummy,1,A.old,9242,5.670141,dummy | A.old,0.000000,17669,0.523063,0.000000
3,dummy,1,B.new,9707,5.679750,dummy | B.new,0.001695,19404,0.500258,-0.043600
4,dummy,2,A.old,7296,7.064796,dummy | A.old,0.000000,17669,0.412927,0.000000
5,dummy,2,B.new,7097,7.078705,dummy | B.new,0.001969,19404,0.365749,-0.114251
6,dummy,3,A.old,6236,8.140194,dummy | A.old,0.000000,17669,0.352935,0.000000
7,dummy,3,B.new,5794,8.077770,dummy | B.new,-0.007669,19404,0.298598,-0.153956
8,dummy,4,A.old,5674,9.055689,dummy | A.old,0.000000,17669,0.321127,0.000000
9,dummy,4,B.new,4906,8.983603,dummy | B.new,-0.007960,19404,0.252834,-0.212666


In [207]:
# hide-output
fig = px.bar(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='cohort_users',
              color='combined_dimension',
              title='Players at each day since install',
              width=1200,
              height=600,
              barmode='group',
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [208]:
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_users',
              color='combined_dimension',
              title='Player level reached at day x since install',
              width=1200,
              height=600,
              hover_data={'pctg_users': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff_users',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              width=1200,
              height=600,
              hover_data={'pctg_diff_users': True, 'cohort_users': True},)

fig.show()

In [209]:


fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='weighted_avg_max_level',
              color='combined_dimension',
              title='Player level reached at day x since install',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

# hide-output
fig = px.line(pl_ftue_max_level_agg, 
              x='days_since_install', 
              y='pctg_diff',
              color='combined_dimension',
              title='Player level reached at day x since install (pct change)',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True, 'cohort_users': True},)

fig.show()

In [210]:
data

,user_id,dt,install_dt,install_dt_week,install_dt_month,days_since_install,max_level,max_gameday,acquisition_type,install_build_version,CPE_flag,FTUE_flag,dummy
0,F0BF22A8C4EE886C,2026-05-31,2026-05-28,2026-05-24,2026-05-01,3,7,4,CPE,0.75.0,Y,A.old,dummy
1,F4E7B68973C9B8C9,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,4,2,CPE,0.75.0,Y,A.old,dummy
2,9446E454237203C7,2026-05-29,2026-05-29,2026-05-24,2026-05-01,0,5,3,CPE,0.75.0,Y,A.old,dummy
3,FB37506040A8CE5F,2026-05-28,2026-05-28,2026-05-24,2026-05-01,0,2,1,CPE,0.75.0,Y,A.old,dummy
4,A8514349B271C50E,2026-05-31,2026-05-29,2026-05-24,2026-05-01,2,7,4,Non-Attributed,0.75.0,N,A.old,dummy
...,...,...,...,...,...,...,...,...,...,...,...,...,...
164819,F574F122CA81CCA6,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,6,4,CPE,0.77.0,Y,B.new,dummy
164820,EA26E8DAA8001B5D,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,3,1,UA,0.77.0,N,B.new,dummy
164821,6608069738F8620F,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,4,2,Non-Attributed,0.77.0,N,B.new,dummy
164822,AF6C23ED8DE9D62,2026-06-21,2026-06-21,2026-06-21,2026-06-01,0,5,3,CPE,0.77.0,Y,B.new,dummy


## Player level reached at dayx percentiles

In [211]:

# hide-output
level_dist = data.groupby(['days_since_install', 'max_level','FTUE_flag']).agg(
    users=('user_id', 'count')
).reset_index()

level_pcts = level_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_level', include_groups=False).reset_index()

fig = px.line(
    level_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_level'),
    x='days_since_install',
    y='max_level',
    color='percentile',
    title='Player level distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
#fig.show()

In [212]:
player_level_agg2 = pl_ftue_max_level_agg[['FTUE_flag', 'days_since_install','weighted_avg_max_level']].drop_duplicates()

# Day-over-day % change in weighted avg level within each install cohort.
# Day 0 is filled as 1.0 (100%) since there is no prior day to compare against.
player_level_agg2['pct_change_weighted_avg_max_level'] = player_level_agg2.groupby('FTUE_flag')['weighted_avg_max_level'].pct_change()
player_level_agg2.pct_change_weighted_avg_max_level = player_level_agg2.pct_change_weighted_avg_max_level.fillna(1)
player_level_agg2

,FTUE_flag,days_since_install,weighted_avg_max_level,pct_change_weighted_avg_max_level
0,A.old,0,3.621509,1.000000
1,B.new,0,3.654821,1.000000
2,A.old,1,5.670141,0.565685
3,B.new,1,5.679750,0.554043
4,A.old,2,7.064796,0.245965
5,B.new,2,7.078705,0.246306
6,A.old,3,8.140194,0.152219
7,B.new,3,8.077770,0.141137
8,A.old,4,9.055689,0.112466
9,B.new,4,8.983603,0.112139


In [213]:
# hide-output
fig = px.line(player_level_agg2.where(player_level_agg2.days_since_install<=30), 
              x='days_since_install', 
              y='pct_change_weighted_avg_max_level',
              color='FTUE_flag',
              title='Player level daily progression by cohort (pct change)',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_level': True})


fig.show()

## Funnel

In [214]:
pl_ftue_funnel_agg = data.groupby(['max_level','FTUE_flag']).agg(
    users=('user_id', 'nunique')
).reset_index()

pl_ftue_funnel_total_agg = pl_ftue_funnel_agg.groupby(['FTUE_flag']).agg(
    total_users=('users', 'sum')
).reset_index()

pl_ftue_funnel_agg = pl_ftue_funnel_agg.merge(pl_ftue_funnel_total_agg, on='FTUE_flag')

pl_ftue_funnel_agg['pctg_users'] = pl_ftue_funnel_agg['users'] / pl_ftue_funnel_agg['total_users']


pl_ftue_funnel_agg['pctg_diff_users'] = pl_ftue_funnel_agg.groupby('max_level')['pctg_users'].pct_change().fillna(0)


pl_ftue_funnel_agg

,max_level,FTUE_flag,users,total_users,pctg_users,pctg_diff_users
0,1,A.old,1978,56834,0.034803,0.000000
1,1,B.new,2203,53991,0.040803,0.172398
2,2,A.old,2359,56834,0.041507,0.000000
3,2,B.new,2398,53991,0.044415,0.070060
4,3,A.old,5134,56834,0.090333,0.000000
...,...,...,...,...,...,...
143,151,A.old,1,56834,0.000018,0.000000
144,151,B.new,1,53991,0.000019,0.052657
145,152,A.old,3,56834,0.000053,0.000000
146,153,A.old,13,56834,0.000229,0.000000


In [215]:
# Define event parameters for level annotations
events_config = {
    'A.old': [
    #    {'level': 7, 'name': 'SP', 'color':'blue'},
    #    {'level': 8, 'name': 'Deco', 'color': 'blue'},
    #    {'level': 10, 'name': 'TimedC', 'color': 'blue'},
    #    {'level': 16, 'name': 'TA', 'color': 'blue'},
    #    {'level': 20, 'name': 'GenB', 'color': 'blue'},
    #    {'level': 25, 'name': 'TgtEvt', 'color': 'blue'},
    ],
    'B.new': [
        {'level': 6, 'name': 'SP', 'color': 'red'},
        {'level': 9, 'name': 'TASign', 'color': 'red'},
        {'level': 10, 'name': 'Deco', 'color': 'red'},
        {'level': 12, 'name': 'TA', 'color': 'red'},
        {'level': 14, 'name': 'GenB', 'color': 'red'},
        {'level': 17, 'name': 'TimedC', 'color': 'red'},
        {'level': 20, 'name': 'TgtEvt', 'color': 'red'},
    ]
}


In [216]:
# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              y='users',
              color='FTUE_flag',
              title='Player level reached at day x since install (pct change)',
              width=1200,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              y='pctg_users',
              color='FTUE_flag',
              title='Player level reached at day x since install (pct change)',
              width=1200,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg)

fig.show()

# hide-output
fig = px.line(pl_ftue_funnel_agg[pl_ftue_funnel_agg['max_level'] <= 30], 
              x='max_level', 
              y='pctg_diff_users',
              color='FTUE_flag',
              title='Player level reached at day x since install (pct change)',
              width=1200,
              height=600,
              hover_data={'max_level': True, 'users': True},)

fig = add_event_annotations(fig, events_config, x_col='max_level', ftue_col='FTUE_flag', data=pl_ftue_funnel_agg)

fig.show()

## Retention

Retention curve: the share of a cohort's total users who were active on each calendar day since install. Plotted on a **log scale** so that differences between cohorts remain visible at longer horizons where absolute percentages are very small. Apr 2024 D1 retention was ~55%; Apr 2026 has declined to ~51%.

In [217]:

retention_data['combined_dimension'] = retention_data['dx'].astype(str) + '_' + retention_data['platform']



retention_data


,install_dt,dx,platform,cohort_size,retained_size,retention_rate,combined_dimension
0,2026-05-08,0,AND,369,369,1.000000,0_AND
1,2026-05-08,0,IOS,931,931,1.000000,0_IOS
2,2026-05-08,1,AND,369,154,0.417344,1_AND
3,2026-05-08,1,IOS,931,345,0.370569,1_IOS
5,2026-05-08,3,AND,369,116,0.314363,3_AND
...,...,...,...,...,...,...,...
350,2026-06-20,0,IOS,1000,1000,1.000000,0_IOS
353,2026-06-20,1,AND,477,204,0.427673,1_AND
352,2026-06-20,1,IOS,1000,372,0.372000,1_IOS
354,2026-06-21,0,AND,478,478,1.000000,0_AND


In [218]:
# hide-output
fig = px.line(retention_data[retention_data['dx'] !=0], 
              x='install_dt', 
              y='retention_rate',
              color='combined_dimension',
              title='Retention rate',
              width=1200,
              height=600,
              hover_data={'install_dt': True, 'retained_size': True},)



fig.show()

In [235]:
retention_data_total

,dx,platform,FTUE_flag,num_cohorts,cohort_size,retained_size,retention_rate
1,0,AND,A.Pre-FTUE revamp,24,7627,7627,1.000000
0,0,AND,B.Post-FTUE revamp,21,9024,9024,1.000000
3,0,IOS,A.Pre-FTUE revamp,22,16035,16035,1.000000
2,0,IOS,B.Post-FTUE revamp,21,17815,17815,1.000000
5,1,AND,A.Pre-FTUE revamp,24,7627,2994,0.392553
4,1,AND,B.Post-FTUE revamp,21,9012,3293,0.365402
6,1,IOS,A.Pre-FTUE revamp,22,16035,6624,0.413096
7,1,IOS,B.Post-FTUE revamp,21,17812,6900,0.387379
8,3,AND,A.Pre-FTUE revamp,24,7627,2066,0.270880
9,3,AND,B.Post-FTUE revamp,21,9012,1932,0.214381


In [236]:
# Prepare data
df_plot = retention_data_total[retention_data_total['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (95% CI based on cohort size)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

In [221]:
# Prepare data
df_plot = retention_data_total_na[retention_data_total_na['dx'] != 0].copy()

# Convert dx to ordered categorical for proper x-axis ordering
dx_vals = sorted(df_plot['dx'].unique())
dx_labels = ['D' + str(d) for d in dx_vals]
df_plot['dx_cat'] = pd.Categorical(
    'D' + df_plot['dx'].astype(str),
    categories=dx_labels,
    ordered=True
)

fig = px.bar(
    df_plot,
    x='dx_cat',
    y='retention_rate',
    color='FTUE_flag',
    text='retention_rate',
    title='Retention rate by FTUE group (95% CI based on cohort size)',
    facet_row='platform',
    barmode='group',
    width=1200,
    height=900,
    hover_data={'retained_size': True, 'cohort_size': True},
    category_orders={'dx_cat': dx_labels}
)

fig.update_traces(texttemplate='%{text:.1%}', textposition='outside')
fig.update_layout(
    yaxis_tickformat='.0%',
    yaxis2_tickformat='.0%',
    uniformtext_minsize=8,
    uniformtext_mode='hide'
)

fig.show()

### Retention Curve

In [222]:
# Count unique users active on each day since install, per cohort
users_agg = pd.DataFrame()
users_agg = data.groupby(['install_dt', 'days_since_install']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Total unique users ever observed in each cohort (denominator for retention %)
users_by_cohort = pd.DataFrame()
users_by_cohort = data.groupby('install_dt').agg(
    total_cohort_users = ('user_id', 'nunique')
).reset_index()

users_agg = users_agg.merge(users_by_cohort, on='install_dt')

# Retention rate: share of cohort still active on a given day
users_agg['pct_cohort_users_active'] = users_agg['unique_users'] / users_agg['total_cohort_users']

users_agg

,install_dt,days_since_install,unique_users,total_cohort_users,pct_cohort_users_active
0,2026-05-08,0,980,988,0.991903
1,2026-05-08,1,487,988,0.492915
2,2026-05-08,2,401,988,0.405870
3,2026-05-08,3,359,988,0.363360
4,2026-05-08,4,328,988,0.331984
...,...,...,...,...,...
523,2026-06-19,1,539,1043,0.516779
524,2026-06-19,2,435,1043,0.417066
525,2026-06-20,0,1073,1073,1.000000
526,2026-06-20,1,546,1073,0.508854


In [223]:
# hide-output

# apply a log transformation to the y axis to better visualize the differences between cohorts, especially in the later days since install where the percentage of active users is very low
fig = px.line(users_agg, 
              x='days_since_install', 
              y='pct_cohort_users_active',
              color='install_dt',
              title='% of cohort users active by days since install (log scale)',
              width=1200,
              height=600,
              hover_data={'pct_cohort_users_active': ':.2%', 'unique_users': True, 'total_cohort_users': True},
              log_y=False)


fig.show()

## Game day reached at day x

Mirrors the level analysis but uses **game days** (in-game calendar progression) instead of levels. Comparing both metrics reveals whether level gates or natural engagement drives pacing — if game days outpace levels, players are replaying content; if levels outpace game days, players are advancing quickly through fewer sessions.

In [224]:
# Same weighted-average approach as player level, applied to max_gameday

# Step 1: Count unique users per (install cohort, day since install, max_gameday bucket)
game_day_agg = data.groupby(['install_dt', 'days_since_install', 'max_gameday']).agg(
    unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 2: Total unique users per cohort-day
daily_cohort_total_users = data.groupby(['install_dt','days_since_install']).agg(
    total_unique_users = ('user_id', 'nunique')
    ).reset_index()

# Step 3: Share of each day's users at each game day value
game_day_agg = game_day_agg.merge(daily_cohort_total_users, on=['install_dt', 'days_since_install'])
game_day_agg['percentage_of_daily_users'] = game_day_agg['unique_users'] / game_day_agg['total_unique_users']

# Step 4: Weighted average game day per cohort-day
weighted_avg = game_day_agg.groupby(['install_dt', 'days_since_install'], group_keys=False).apply(
    lambda x: (x['max_gameday'] * x['unique_users']).sum() / x['unique_users'].sum(),
    include_groups=False
).reset_index()

weighted_avg.columns = ['install_dt', 'days_since_install', 'weighted_avg_max_gameday']
game_day_agg = game_day_agg.merge(weighted_avg, on=['install_dt', 'days_since_install'])

# Drop small buckets (< 50 users) to reduce noise
game_day_agg = game_day_agg[game_day_agg['unique_users'] >= 50]

# Collapse to one row per cohort-day
game_day_agg = game_day_agg.groupby(['install_dt', 'days_since_install']).agg(
    cohort_users = ('unique_users', 'sum'),
    weighted_avg_max_gameday = ('weighted_avg_max_gameday', 'first')
).reset_index()

game_day_agg

,install_dt,days_since_install,cohort_users,weighted_avg_max_gameday
0,2026-05-08,0,948,1.954082
1,2026-05-08,1,361,3.484600
2,2026-05-08,2,269,4.763092
3,2026-05-08,3,182,5.623955
4,2026-05-08,4,159,6.368902
...,...,...,...,...
244,2026-06-19,1,437,3.319109
245,2026-06-19,2,315,4.367816
246,2026-06-20,0,1024,2.022367
247,2026-06-20,1,422,3.664835


In [225]:
# hide-output
fig = px.line(game_day_agg, 
              x='days_since_install', 
              y='weighted_avg_max_gameday',
              color='install_dt',
              title='Game day daily progression by cohort',
              width=1200,
              height=600,
              hover_data={'weighted_avg_max_gameday': True, 'cohort_users': True},
              )


fig.show()

In [226]:
# hide-output
gameday_dist = data.groupby(['days_since_install', 'max_gameday']).agg(
    users=('user_id', 'count')
).reset_index()

gameday_pcts = gameday_dist.groupby('days_since_install').apply(weighted_quantiles, measure_col='max_gameday', include_groups=False).reset_index()

fig = px.line(
    gameday_pcts.melt(id_vars='days_since_install', var_name='percentile', value_name='max_gameday'),
    x='days_since_install',
    y='max_gameday',
    color='percentile',
    title='Game day distribution by days since install (P10 / P50 / P90)',
    width=1200,
    height=600,
)
fig.show()

# Things to do next
- Is the churn increasing for P90 players when they reach the 150 GD mark?
- Is the slowdown on pace seen on around level 50 on newest cohort caused by CPE mix? what does it look like with only organics?